# Tạo giọng thuyết minh — *Sparse Feature Circuits*

Bật GPU: **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**. Xong sẽ tự tải về `tts_mp3.zip`.

In [ ]:
!pip -q install zerotts edge-tts soundfile

In [ ]:
LINES = {
 "intro_0": "Chào bạn! Hôm nay mình cùng mổ xẻ một paper về AI có thể giải thích được, đăng ở hội nghị ai xi eo a hai nghìn không trăm hai mươi lăm.",
 "intro_1": "Tên paper là Sparse Feature Circuits, tạm dịch: các mạch đặc trưng thưa.",
 "intro_2": "Nghe thì đáng sợ, nhưng mình sẽ giải thích từng khái niệm một, trước khi dùng nó.",
 "lm_0": "Trước hết, mô hình ngôn ngữ là gì? Đó là một chương trình đoán từ tiếp theo trong câu.",
 "lm_1": "Ví dụ, sau câu: The boys near the teacher..., mô hình phải chọn giữa have và has.",
 "lm_2": "Chủ ngữ là boys, số nhiều, nên đáp án đúng là have. Mô hình thường chọn đúng.",
 "lm_3": "Nhưng câu hỏi là: bên trong nó đã làm gì để ra được quyết định đó?",
 "lm_4": "Bên trong mô hình chỉ là hàng triệu con số chạy qua nhiều tầng. Nhìn vào đó, con người chẳng hiểu gì cả. Đây gọi là vấn đề hộp đen.",
 "lm_5": "Lĩnh vực ích ây ai, hay AI giải thích được, cố gắng mở chiếc hộp đen này ra.",
 "circuit_def_0": "Một cách mở hộp đen là tìm mạch, hay circuit.",
 "circuit_def_1": "Mạch là một nhóm nhỏ các bộ phận bên trong mô hình, cùng các đường nối giữa chúng, đủ để giải thích một hành vi cụ thể.",
 "circuit_def_2": "Giống như trong một thành phố khổng lồ, ta chỉ cần vẽ đúng những con đường mà chiếc xe đã đi qua.",
 "problem_0": "Các nghiên cứu trước đây vẽ mạch bằng những bộ phận thô, như neuron hay attention head.",
 "problem_1": "Vấn đề thứ nhất: một neuron thường đa nghĩa. Cùng một neuron có thể sáng lên với con mèo, với xe hơi, lẫn tiếng Pháp.",
 "problem_2": "Vậy nên dù tìm ra mạch, ta vẫn không hiểu mạch đó đang làm gì.",
 "problem_3": "Vấn đề thứ hai: nếu chia nhỏ ra thành đơn vị tinh hơn thì có hàng chục nghìn đơn vị. Thử từng cái một thì tốn thời gian khủng khiếp.",
 "problem_4": "Paper này giải quyết cả hai vấn đề: dùng Sparse Autoencoder để có đơn vị dễ hiểu, và dùng xấp xỉ tuyến tính để tính cho nhanh.",
 "vector_0": "Trước khi vào Sparse Autoencoder, mình ôn nhanh khái niệm vector.",
 "vector_1": "Vector đơn giản là một danh sách các con số. Với hai con số, ta vẽ nó thành một mũi tên trên mặt phẳng.",
 "vector_2": "Hai vector có thể cộng lại: đặt mũi tên nối đuôi nhau, ta được một mũi tên mới.",
 "vector_3": "Và ta có thể nhân vector với một số, gọi là hệ số, để kéo dài hoặc thu ngắn nó.",
 "vector_4": "Trong mô hình ngôn ngữ, trạng thái bên trong tại mỗi từ là một vector rất dài, gọi là activation, ký hiệu x. Với Pythia, nó có 512 con số.",
 "sae_0": "Ý tưởng của Sparse Autoencoder, viết tắt là ét a e, như sau.",
 "sae_1": "Ta chuẩn bị sẵn một cuốn từ điển gồm rất nhiều vector hướng, mỗi hướng là một feature, hay đặc trưng, ký hiệu v i.",
 "sae_2": "Rồi ta biểu diễn activation x thành tổng của một vài feature, mỗi cái nhân với một hệ số f i.",
 "sae_3": "Chữ sparse, nghĩa là thưa, tức là hầu hết hệ số bằng 0. Mỗi lần chỉ vài feature được bật lên.",
 "sae_4": "Viết thành công thức: x bằng tổng các f i nhân v i, cộng b, cộng epsilon.",
 "sae_5": "Ký hiệu Sigma nghĩa là cộng dồn qua tất cả các feature. b là độ lệch cố định. Còn epsilon là phần sai số mà từ điển chưa diễn tả được.",
 "sae_6": "Điều kỳ diệu là các feature này thường dễ hiểu với con người, ví dụ: feature bật lên khi gặp danh từ số nhiều.",
 "sae_train_0": "ét a e học cuốn từ điển đó bằng cách tối ưu hai mục tiêu cùng lúc.",
 "sae_train_1": "Mục tiêu một: bản tái tạo x mũ phải gần với x thật, tức là sai số bình phương nhỏ.",
 "sae_train_2": "Mục tiêu hai: phạt việc dùng quá nhiều feature, để các hệ số f i luôn thưa.",
 "sae_train_3": "Từ điển rất rộng: với Pythia, số feature gấp 64 lần số chiều. Với Gemma 2, tác giả dùng bộ ét a e có sẵn tên Gemma Scope.",
 "graph_0": "Và đây là mẹo quan trọng đầu tiên của paper.",
 "graph_1": "Ta chèn ét a e vào nhiều vị trí trong mô hình: attention, em eo pi và dòng residual ở mỗi tầng.",
 "graph_2": "Khi đó, mỗi feature, và cả phần sai số epsilon, trở thành một nút trong đồ thị tính toán. Hình vuông là feature, hình tam giác là sai số.",
 "graph_3": "Việc tìm mạch bây giờ chỉ là: chọn ra những nút và cạnh quan trọng nhất trong đồ thị này.",
 "metric_0": "Nhưng quan trọng với cái gì? Ta cần một thước đo, ký hiệu m.",
 "metric_1": "Mô hình gán cho mỗi từ một xác suất. Ta lấy log xác suất của have trừ đi log xác suất của has.",
 "metric_2": "Nếu m dương và lớn, mô hình nghiêng mạnh về have. Nếu âm, nó nghiêng về has.",
 "metric_3": "Log chỉ là một cách co giãn thang đo cho dễ so sánh, bạn không cần lo về nó.",
 "ie_0": "Để đo tầm quan trọng của một nút a, paper dùng khái niệm hiệu ứng gián tiếp, gọi tắt là ai i.",
 "ie_1": "Ta có hai câu gần giống nhau: câu sạch, The teacher, và câu vá, The teachers. Chỉ khác số ít, số nhiều.",
 "ie_2": "Chạy mô hình trên câu sạch. Rồi, đúng tại nút a, ta thay giá trị của nó bằng giá trị lấy từ câu vá. Ký hiệu do nghĩa là: ép nút này nhận giá trị đó.",
 "ie_3": "ai i bằng thước đo m sau khi can thiệp, trừ đi m ban đầu. Nếu m thay đổi nhiều, nút a thật sự có ảnh hưởng.",
 "ie_4": "Đây là suy luận nhân quả: ta không chỉ nhìn tương quan, mà thật sự can thiệp và xem kết quả thay đổi.",
 "cost_0": "Vấn đề là: làm như vậy thì mỗi nút cần chạy lại cả mô hình một lần.",
 "cost_1": "Với hàng chục nghìn feature ở nhiều vị trí, việc này quá chậm.",
 "atp_0": "Giải pháp là xấp xỉ tuyến tính, dùng gradient. Gradient nghĩa là độ dốc: nếu nhích a một chút, m thay đổi bao nhiêu.",
 "atp_1": "Hãy tưởng tượng m là một đường cong theo a. Tại điểm sạch, ta vẽ đường tiếp tuyến.",
 "atp_2": "Thay vì thật sự đi tới điểm vá, ta đi dọc theo tiếp tuyến: độ dốc nhân với khoảng cách, là ra ước lượng.",
 "atp_3": "Đó là công thức attribution patching: ai i xấp xỉ bằng gradient của m theo a, nhân với a vá trừ a sạch.",
 "atp_4": "Cái hay: chỉ cần hai lượt chạy xuôi và một lượt lan truyền ngược, ta có ước lượng cho tất cả các nút cùng lúc.",
 "atp_5": "Lan truyền ngược, hay backpropagation, là thuật toán tính gradient cho mọi nút chỉ trong một lần.",
 "ig_0": "Nhưng nếu đường cong uốn mạnh, tiếp tuyến sẽ đoán sai, nhất là ở các tầng đầu của mô hình.",
 "ig_1": "Khi đó paper dùng integrated gradients, tức là gradient tích phân.",
 "ig_2": "Ta chia đoạn từ a sạch tới a vá thành 10 bước, đo độ dốc tại mỗi bước, rồi lấy trung bình.",
 "ig_3": "Độ dốc trung bình này bám theo đường cong tốt hơn nhiều, đổi lại tốn thêm khoảng 10 lần tính toán.",
 "filter_0": "Có điểm ai i cho mọi nút rồi, ta lấy trung bình trên nhiều cặp câu.",
 "filter_1": "Sau đó giữ lại những nút có độ lớn ai i vượt ngưỡng tê en. Các nút còn lại bị bỏ đi.",
 "filter_2": "Làm tương tự với các cạnh nối giữa các nút, dùng ngưỡng tê i.",
 "filter_3": "Kết quả là một đồ thị nhỏ gọn: một mạch đặc trưng thưa, gồm các feature dễ hiểu và cách chúng truyền thông tin cho nhau.",
 "pipeline_0": "Tóm lại, quy trình có bốn bước.",
 "pipeline_1": "Bước một: chạy mô hình trên câu sạch và câu vá, lưu lại mọi activation.",
 "pipeline_2": "Bước hai: lan truyền ngược, lưu gradient của m tại mọi nút.",
 "pipeline_3": "Bước ba: tính ai i xấp xỉ cho từng nút, lọc theo ngưỡng.",
 "pipeline_4": "Bước bốn: tính ai i cho các cạnh, lọc tiếp. Và ta có mạch.",
 "eval_0": "Làm sao biết mạch này tốt? Paper dùng hai thước đo.",
 "eval_1": "Thứ nhất là độ trung thành, faithfulness. Ta chỉ giữ lại mạch C, còn mọi nút khác bị thay bằng giá trị trung bình. Gọi là mean ablation.",
 "eval_2": "Faithfulness bằng m của mạch trừ m của mạch rỗng, chia cho m của cả mô hình trừ m của mạch rỗng. Tức là: mạch giải thích được bao nhiêu phần trăm hành vi. Lý tưởng là 1.",
 "eval_3": "Thứ hai là độ đầy đủ, completeness: bỏ mạch đi, xem phần còn lại còn làm được việc không. Lý tưởng là gần 0.",
 "eval_4": "Kết quả: trên Pythia, chỉ khoảng 100 feature đã giải thích phần lớn hành vi, trong khi cần khoảng 1500 neuron mới giải thích được một nửa.",
 "eval_5": "Trên Gemma 2, khoảng 500 feature so với khoảng 50 nghìn neuron. Feature hiệu quả hơn hẳn.",
 "case_0": "Giờ ta xem một mạch cụ thể: hòa hợp chủ ngữ và động từ qua mệnh đề quan hệ.",
 "case_1": "Câu mẫu: The girl that the teacher sees, rồi đến has hoặc have. Chú ý: động từ phải hợp với girl, không phải teacher.",
 "case_2": "Đầu tiên, các feature ở tầng thấp phát hiện số ít hay số nhiều của danh từ girl.",
 "case_3": "Tiếp theo, các feature nhận ra từ that đang mở đầu một mệnh đề phụ.",
 "case_4": "Rồi các feature phát hiện mệnh đề phụ đã kết thúc, và mang thông tin số của chủ ngữ tới vị trí cuối.",
 "case_5": "Cuối cùng, các feature phân biệt dạng động từ đẩy xác suất về has hoặc have.",
 "case_6": "Mạch này đọc được như một thuật toán mà con người hiểu. Gemma 2 còn dùng thêm các feature theo dõi số của cả cụm danh từ.",
 "shift_problem_0": "Phần ứng dụng thú vị nhất: Shift, tức là cắt tỉa các feature giả tạo mà con người hiểu được.",
 "shift_problem_1": "Bài toán: phân loại nghề nghiệp từ tiểu sử, giáo sư hay y tá.",
 "shift_problem_2": "Nhưng dữ liệu huấn luyện bị lệch cực độ: toàn giáo sư nam và y tá nữ.",
 "shift_problem_3": "Mô hình sẽ đi đường tắt: đoán theo giới tính thay vì nghề nghiệp. Đó là tín hiệu giả, spurious signal.",
 "shift_problem_4": "Các phương pháp cũ cần thêm dữ liệu có nhãn để gỡ lệch. Ở đây, ta không có dữ liệu đó.",
 "shift_method_0": "Shift làm như sau. Bước một: tìm mạch đặc trưng giải thích quyết định của bộ phân loại.",
 "shift_method_1": "Bước hai: con người đọc từng feature trong mạch, và đánh giá nó có liên quan đến nghề nghiệp hay không.",
 "shift_method_2": "Ví dụ, feature bật lên với từ she, her, hay women là về giới tính, không liên quan tới nghề.",
 "shift_method_3": "Bước ba: tắt hẳn các feature không liên quan, bằng cách đặt chúng về 0. Bước bốn, tùy chọn: huấn luyện lại lớp phân loại.",
 "shift_result_0": "Kết quả trên Pythia, đo trên dữ liệu cân bằng.",
 "shift_result_1": "Độ chính xác đoán nghề tăng từ 62 phần trăm lên 88, và lên 93 sau khi huấn luyện lại, ngang với mô hình lý tưởng.",
 "shift_result_2": "Khả năng đoán giới tính giảm từ 87 xuống khoảng 52 phần trăm, gần như đoán mò. Nghĩa là thiên kiến giới tính đã gần như biến mất.",
 "shift_result_3": "Nhóm tệ nhất tăng từ 24 lên 89 phần trăm. Còn nếu tắt feature ngẫu nhiên, hay dùng neuron, thì không đạt được như vậy.",
 "unsup_0": "Cuối cùng, paper tự động hóa toàn bộ quy trình ở quy mô lớn.",
 "unsup_1": "Lấy một kho văn bản khổng lồ, gom các ví dụ có cách dự đoán giống nhau thành từng cụm, không cần con người gán nhãn.",
 "unsup_2": "Mỗi cụm là một hành vi, ví dụ: đếm tiếp dãy số một, hai, ba, bốn, hoặc dự đoán từ to sau các động từ.",
 "unsup_3": "Rồi tìm mạch cho từng cụm. Kết quả là hàng nghìn mạch cho hàng nghìn hành vi, có thể xem tại trang feature circuits chấm ích i dét.",
 "limits_0": "Tất nhiên, phương pháp vẫn có hạn chế.",
 "limits_1": "Cần có ét a e được huấn luyện sẵn, tốn nhiều tài nguyên tính toán ban đầu.",
 "limits_2": "Phần sai số epsilon vẫn là hộp đen nhỏ chưa giải thích được.",
 "limits_3": "Và việc gán ý nghĩa cho feature phần lớn vẫn là đánh giá định tính của con người.",
 "outro_0": "Tóm lại: paper dùng ét a e để có những đơn vị dễ hiểu, dùng gradient để tìm nhanh những đơn vị quan trọng, và nối chúng thành mạch.",
 "outro_1": "Mạch đó giúp ta hiểu mô hình, và còn giúp ta sửa mô hình, như gỡ bỏ thiên kiến giới tính mà không cần thêm dữ liệu.",
 "outro_2": "Cảm ơn bạn đã theo dõi!"
}
print(len(LINES), 'câu')

In [ ]:
ENGINE = 'zerotts'          # 'zerotts' hoặc 'edge'
VOICE  = 'quangminh'        # ZeroTTS: tên giọng | Edge: 'vi-VN-HoaiMyNeural' / 'vi-VN-NamMinhNeural'
EDGE_RATE = '+0%'

import os, subprocess, shutil, numpy as np, soundfile as sf
from tqdm.auto import tqdm
from IPython.display import Audio, display

if ENGINE == 'zerotts':
    from zerotts import ZeroTTS
    try:
        from zerotts import normalize_vi_text
    except Exception:
        normalize_vi_text = lambda s: s
    tts = ZeroTTS.from_pretrained("zeroweight-ai/ZeroTTS")
    SR = getattr(tts, 'sample_rate', 48000)
    def zt_synth(text, voice, path_wav):
        audio = tts.synthesize(normalize_vi_text(text), voice=voice)
        try:
            tts.save_audio(audio, path_wav)
        except Exception:
            a = np.asarray(audio.detach().cpu() if hasattr(audio, 'detach') else audio).squeeze()
            sf.write(path_wav, a, SR)
else:
    import edge_tts
    async def edge_synth(text, voice, path, rate='+0%'):
        await edge_tts.Communicate(text, voice, rate=rate).save(path)

os.makedirs('tts_mp3', exist_ok=True)
failed = []
for k, text in tqdm(LINES.items()):
    out = f'tts_mp3/{k}.mp3'
    if os.path.exists(out): continue
    try:
        if ENGINE == 'zerotts':
            zt_synth(text, VOICE, f'tts_mp3/{k}.wav')
            subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', f'tts_mp3/{k}.wav', '-ac', '1', '-b:a', '128k', out], check=True)
            os.remove(f'tts_mp3/{k}.wav')
        else:
            await edge_synth(text, VOICE, out, EDGE_RATE)
    except Exception as e:
        failed.append((k, str(e)))
print('Lỗi:', failed if failed else 'không có')
print(len([f for f in os.listdir('tts_mp3') if f.endswith('.mp3')]), '/', len(LINES), 'file')
display(Audio('tts_mp3/intro_0.mp3'))

shutil.make_archive('tts_mp3', 'zip', 'tts_mp3')
from google.colab import files
files.download('tts_mp3.zip')

Bỏ `tts_mp3.zip` vào thư mục **xai/video** rồi nhắn Claude để gắn vào video.